### Allscripts Sunrise (SCM) Fact Relationship Hydration

Creates bidirectional OMOP fact relationships between populated SCM clinical facts and `visit_occurrence`.

Notes:
- Reset is active at the top for clean reruns.
- This notebook only builds visit-linked fact relationships.
- Domains with no populated rows simply contribute no inserts.

In [ ]:
%sql
TRUNCATE TABLE _exponent.omop_scm.fact_relationship;

In [ ]:
%sql
INSERT INTO _exponent.omop_scm.fact_relationship (
  fact_id_1,
  domain_concept_id_1,
  fact_id_2,
  domain_concept_id_2,
  relationship_concept_id
)
WITH domain_ids AS (
  SELECT domain_id, domain_concept_id
  FROM _exponent.omop.domain
  WHERE domain_id IN ('Visit', 'Condition', 'Drug', 'Measurement', 'Observation', 'Procedure', 'Device')
), fact_links AS (
  SELECT condition_occurrence_id AS fact_id_1, 'Condition' AS domain_1, visit_occurrence_id AS fact_id_2, 'Visit' AS domain_2
  FROM _exponent.omop_scm.condition_occurrence
  WHERE condition_occurrence_id IS NOT NULL AND visit_occurrence_id IS NOT NULL
  UNION ALL
  SELECT drug_exposure_id AS fact_id_1, 'Drug' AS domain_1, visit_occurrence_id AS fact_id_2, 'Visit' AS domain_2
  FROM _exponent.omop_scm.drug_exposure
  WHERE drug_exposure_id IS NOT NULL AND visit_occurrence_id IS NOT NULL
  UNION ALL
  SELECT measurement_id AS fact_id_1, 'Measurement' AS domain_1, visit_occurrence_id AS fact_id_2, 'Visit' AS domain_2
  FROM _exponent.omop_scm.measurement
  WHERE measurement_id IS NOT NULL AND visit_occurrence_id IS NOT NULL
  UNION ALL
  SELECT observation_id AS fact_id_1, 'Observation' AS domain_1, visit_occurrence_id AS fact_id_2, 'Visit' AS domain_2
  FROM _exponent.omop_scm.observation
  WHERE observation_id IS NOT NULL AND visit_occurrence_id IS NOT NULL
  UNION ALL
  SELECT procedure_occurrence_id AS fact_id_1, 'Procedure' AS domain_1, visit_occurrence_id AS fact_id_2, 'Visit' AS domain_2
  FROM _exponent.omop_scm.procedure_occurrence
  WHERE procedure_occurrence_id IS NOT NULL AND visit_occurrence_id IS NOT NULL
  UNION ALL
  SELECT device_exposure_id AS fact_id_1, 'Device' AS domain_1, visit_occurrence_id AS fact_id_2, 'Visit' AS domain_2
  FROM _exponent.omop_scm.device_exposure
  WHERE device_exposure_id IS NOT NULL AND visit_occurrence_id IS NOT NULL
)
SELECT DISTINCT
  fl.fact_id_1,
  d1.domain_concept_id,
  fl.fact_id_2,
  d2.domain_concept_id,
  33136 AS relationship_concept_id
FROM fact_links fl
JOIN domain_ids d1 ON d1.domain_id = fl.domain_1
JOIN domain_ids d2 ON d2.domain_id = fl.domain_2
UNION ALL
SELECT DISTINCT
  fl.fact_id_2,
  d2.domain_concept_id,
  fl.fact_id_1,
  d1.domain_concept_id,
  33137 AS relationship_concept_id
FROM fact_links fl
JOIN domain_ids d1 ON d1.domain_id = fl.domain_1
JOIN domain_ids d2 ON d2.domain_id = fl.domain_2;

In [ ]:
%sql
SELECT
  relationship_concept_id,
  domain_concept_id_1,
  domain_concept_id_2,
  COUNT(*) AS row_count
FROM _exponent.omop_scm.fact_relationship
GROUP BY relationship_concept_id, domain_concept_id_1, domain_concept_id_2
ORDER BY row_count DESC;